# GHIST+ improvements on free Kaggle compute

**Settings:** Internet *on*. Use *CPU* for Part A. Use a Kaggle GPU only for Part B after its prerequisites are present. Add the private Dataset **ghist-adapt-code** via *Add Data*.

- **Part A (CPU, expected ~2–3 h):** P4a re-tests zero-shot soft output (P1), HAC vs the authors' RRC (P1), trust gate (P2), and uncertainty (P3) on the released pan-cancer GHIST+ model and Breast5K slide, using the unchanged protocols (`config_v4/5/7.json`).
- **Part B (GPU):** P4b uses UNI2-h image features for image-aware HAC. It runs only when a Kaggle Secret named `HF_TOKEN` is available and direct H&E plus alignment URLs are set in `dgx/he_sources.json`; otherwise Part B cells report that they were skipped.


In [ ]:
import glob, importlib.util, os, shutil, subprocess, sys, zipfile
project = '/kaggle/working/GHIST_plus'
os.makedirs(project, exist_ok=True)
roots = glob.glob('/kaggle/input/datasets/*/ghist-adapt-code') + glob.glob('/kaggle/input/ghist-adapt-code')
root = next((r for r in roots if os.path.isfile(os.path.join(r, 'reproduction/ghist_adapt/p1_run.py'))), None)
if root:
    shutil.copytree(root, project, dirs_exist_ok=True)
else:
    archives = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    assert archives, 'Attach ghist-adapt-code as a Kaggle Dataset first'
    zipfile.ZipFile(archives[0]).extractall(project)
assert os.path.isfile(os.path.join(project, 'reproduction/ghist_adapt/p1_run.py')), 'GHIST+ code bundle did not unpack correctly'
os.environ.update(GHIST_BUNDLE_ROOT='/tmp/bundle', GHIST_V3_FETCH='/tmp/bundle', GHIST_ADAPT_CACHE='/tmp/cache',
                  OPENBLAS_NUM_THREADS='4', OMP_NUM_THREADS='4')
packages = {'numpy': 'numpy==1.26.4', 'pandas': 'pandas==2.3.2', 'tabulate': 'tabulate',
            'timm': 'timm==0.9.12', 'huggingface_hub': 'huggingface_hub'}
missing = [pkg for mod, pkg in packages.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
%cd /kaggle/working/GHIST_plus/reproduction/ghist_adapt
!nproc; free -g; df -h /tmp | tail -1


## Part A: second-checkpoint confirmation (CPU)

In [ ]:
from huggingface_hub import hf_hub_download
REV = '37a937677c5a8ebe5c7b8fbd486fbc5bb50ce1de'
for f in ['GHIST_plus/predictions/pancancer/breast_5k/test_slide11_pred_expr_scaled.npz',
          'evaluation_data/breast5k/cell_gene_matrix_filtered.csv',
          'evaluation_data/breast5k/cell_coords_histology_filtered.csv']:
    print(hf_hub_download('SydneyBioX/GHIST-Plus-bundle', f, repo_type='dataset', revision=REV, local_dir='/tmp/bundle'))

In [ ]:
!python slides.py breast5k_pan && python hurdle.py breast5k_pan
!python p1_run.py breast5k_pan 2>&1 | tail -12

In [ ]:
!python p2_run.py breast5k_pan 2>&1 | tail -10
!python p3_run.py breast5k_pan 2>&1 | tail -7

In [ ]:
import json, pandas as pd
z = json.load(open('outputs_v4/breast5k_pan/zero_shot.json'))
print('zero-shot, all cells:', {k: round(v['median_pcc_all'], 4) for k, v in z.items()})
r = pd.read_csv('outputs_v4/breast5k_pan/runs.csv'); r['cfg'] = r.config.str.replace(r'_s\d+$', '', regex=True)
display(r.groupby(['cfg','method']).median_pcc_all.mean().unstack().round(4))
!cd /kaggle/working && zip -qr ghist_adapt_results_P4a.zip GHIST_plus/reproduction/ghist_adapt/outputs_v4 GHIST_plus/reproduction/ghist_adapt/outputs_v5 GHIST_plus/reproduction/ghist_adapt/outputs_v7 && ls -lh ghist_adapt_results_P4a.zip

## Part B: UNI2-h image-aware HAC (GPU)
1. Paste the H&E OME-TIFF and alignment CSV links into `dgx/he_sources.json` (or edit the dict below).
2. Run the QA cell and **look at the image**: the green dots must sit on nuclei.
3. Only then extract, set `frozen` in `config_v6.json`, and run P4b.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient
try:
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    hf_token = ''
os.environ['HF_TOKEN'] = hf_token
source = json.load(open('dgx/he_sources.json'))['breast5k']
he_url = source.get('he_ome_tif_url', '')
align_url = source.get('alignment_csv_url', '')
part_b_ready = bool(hf_token and he_url and align_url)
if part_b_ready:
    subprocess.run([sys.executable, 'dgx/fetch_he.py', 'breast5k'], check=True)
    subprocess.run([sys.executable, 'dgx/extract_uni2h.py', '--he', '/tmp/bundle/he/breast5k/he.ome.tif',
                    '--align', '/tmp/bundle/he/breast5k/he_alignment.csv',
                    '--cells', '/tmp/bundle/evaluation_data/breast5k/cell_coords_histology_filtered.csv',
                    '--cell-units', 'um', '--out', '/tmp/emb/breast5k_uni2h', '--qa-only'], check=True)
    from IPython.display import Image, display
    display(Image('/tmp/emb/breast5k_uni2h_qa.png'))
else:
    print('Part B skipped: requires Kaggle Secret HF_TOKEN and direct H&E/alignment URLs in dgx/he_sources.json.')


In [ ]:
if part_b_ready:
    subprocess.run([sys.executable, 'dgx/extract_uni2h.py', '--he', '/tmp/bundle/he/breast5k/he.ome.tif',
                    '--align', '/tmp/bundle/he/breast5k/he_alignment.csv',
                    '--cells', '/tmp/bundle/evaluation_data/breast5k/cell_coords_histology_filtered.csv',
                    '--cell-units', 'um', '--out', '/tmp/emb/breast5k_uni2h', '--batch', '256', '--workers', '4'], check=True)
else:
    print('Part B full feature extraction skipped because its prerequisites are not configured.')


In [ ]:
if part_b_ready:
    import datetime
    c = json.load(open('config_v6.json'))
    c['frozen'] = datetime.datetime.utcnow().isoformat() + 'Z (Kaggle, after QA, before any P4 metric)'
    json.dump(c, open('config_v6.json', 'w'), indent=2)
    subprocess.run([sys.executable, 'dgx/p4_run.py', 'breast5k', '--emb', '/tmp/emb/breast5k_uni2h'], check=True)
    subprocess.run(['zip', '-qr', '/kaggle/working/ghist_adapt_results_P4b.zip',
                    'GHIST_plus/reproduction/ghist_adapt/outputs_v6',
                    'GHIST_plus/reproduction/ghist_adapt/config_v6.json'], check=True)
else:
    print('P4b skipped; run the Part B prerequisite cell after adding access and files.')
